# Прогнозен изкуствен интелект (Predictive AI)
## Предсказване на цени на апартаменти в София с FastAI

### Описание

**Dataset:** [Bulgaria Real Estate Listings](https://www.kaggle.com/datasets/gabrielagencheva/bulgaria-real-estate-listings) от Kaggle. Съдържа реални обяви за имоти от imot.bg. За целите на проекта филтрираме само **обяви за продажба на апартаменти в София** — около **22 000 записа** с характеристики като площ (m2), тип апартамент (1-стаен, 2-стаен, мезонет и др.), тип строителство (тухла, панел и др.), етаж, брой етажи, година на строеж, квартал, цена на м² и 46 вида удобства (асансьор, паркинг, климатик, обзавеждане и др.). Целевата променлива е **цената на апартамента в EUR**.

**Модел:** Използваме FastAI `tabular_learner` — напълно свързана невронна мрежа за таблични данни с:
- Embedding слоеве за категорийни променливи (квартал, тип апартамент, тип строителство)
- Два скрити слоя с 400 и 200 неврона
- Batch Normalization и Dropout 30% за регуларизация
- Weight decay 0.1 за допълнителна регуларизация
- Log трансформация на целевата променлива
- OneCycleLR learning rate scheduler

**Резултат:** Моделът предсказва продажната цена на апартамент в София (в EUR) по неговите характеристики. Оценката се измерва чрез RMSE, MAE и R² Score върху валидационното множество.

## 1. Инсталиране и импорт на библиотеки

In [ ]:
# Инсталиране (разкоментирай при нужда)
# !pip install fastai kaggle pandas matplotlib scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from fastai.tabular.all import *
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.max_columns', 30)
print('Библиотеките са заредени успешно!')

## 2. Зареждане на данните от Kaggle

Dataset-ът [Bulgaria Real Estate Listings](https://www.kaggle.com/datasets/gabrielagencheva/bulgaria-real-estate-listings) съдържа няколко свързани CSV таблици. Ние ги обединяваме и филтрираме само обявите за **продажба на апартаменти в София**.

**Вариант А:** Kaggle API — изтегляне автоматично.

**Вариант Б:** Ръчно — изтегли от https://www.kaggle.com/datasets/gabrielagencheva/bulgaria-real-estate-listings и разархивирай в папка `data_sofia/`.

In [ ]:
import os

data_path = Path('data_sofia')
clean_path = data_path / 'clean'

if not clean_path.exists():
    data_path.mkdir(exist_ok=True)
    try:
        !kaggle datasets download -d gabrielagencheva/bulgaria-real-estate-listings -p data_sofia/ --unzip
        print('Данните са изтеглени от Kaggle!')
    except Exception as e:
        print(f'Не може да се изтегли автоматично: {e}')
        print('Моля изтегли ръчно от:')
        print('https://www.kaggle.com/datasets/gabrielagencheva/bulgaria-real-estate-listings')
else:
    print('Данните вече съществуват в data_sofia/')

In [ ]:
# Зареждане на таблиците
listings = pd.read_csv(clean_path / 'listings.csv')
properties = pd.read_csv(clean_path / 'properties.csv')
geographies = pd.read_csv(clean_path / 'geographies.csv')
property_types = pd.read_csv(clean_path / 'property_types.csv')
construction_types = pd.read_csv(clean_path / 'construction_types.csv')
features = pd.read_csv(clean_path / 'features.csv')
property_features = pd.read_csv(clean_path / 'property_features.csv')

print(f'Обяви: {len(listings)}')
print(f'Имоти: {len(properties)}')
print(f'Географии: {len(geographies)}')
print(f'Типове имоти: {len(property_types)}')
print(f'Типове строителство: {len(construction_types)}')
print(f'Удобства: {len(features)}')
print(f'Имот-удобства връзки: {len(property_features)}')

## 3. Филтриране на данните за апартаменти в София и обединяване

In [ ]:
# Намиране на всички geo_id принадлежащи към София
sofia_region = geographies[(geographies['name_bg'] == 'София') & (geographies['level'] == 'region')]
sofia_region_id = sofia_region['geo_id'].values[0]
print(f'Sofia region geo_id: {sofia_region_id}')

sofia_ids = set([sofia_region_id])

# Локалитети под София
localities = geographies[geographies['parent_id'] == float(sofia_region_id)]
sofia_ids.update(localities['geo_id'].tolist())

# Квартали/райони под локалитетите
for lid in localities['geo_id'].tolist():
    areas = geographies[geographies['parent_id'] == float(lid)]
    sofia_ids.update(areas['geo_id'].tolist())

print(f'Общо географски зони в София: {len(sofia_ids)}')

In [ ]:
# Филтриране на имоти в София
sofia_props = properties[properties['geo_id'].isin(sofia_ids)].copy()

# Обединяване с обявите (само продажби с цена)
df = sofia_props.merge(
    listings[['property_id', 'price', 'transaction_type']],
    on='property_id'
)
df = df[(df['transaction_type'] == 'sale') & (df['price'].notna()) & (df['price'] > 0)].copy()

# Добавяне на тип имот
df = df.merge(property_types[['property_type_id', 'name_en', 'category']], on='property_type_id', how='left')
df.rename(columns={'name_en': 'property_type', 'category': 'property_category'}, inplace=True)

# Филтриране САМО на апартаменти
apartment_types = ['apartment', 'multi-room apartment', 'maisonette', 'studio/attic']
df = df[df['property_type'].isin(apartment_types)].copy()
print(f'Апартаменти в София: {len(df)}')

# Добавяне на тип строителство
df = df.merge(construction_types[['construction_type_id', 'name_en']], on='construction_type_id', how='left')
df.rename(columns={'name_en': 'construction_type'}, inplace=True)

# Добавяне на квартал
df = df.merge(geographies[['geo_id', 'name_bg']], on='geo_id', how='left')
df.rename(columns={'name_bg': 'neighborhood'}, inplace=True)

print(f'\nКолони: {df.columns.tolist()}')

In [ ]:
# Pivot на удобствата — всяко удобство става отделна колона (0/1)
pf_sofia = property_features[property_features['property_id'].isin(df['property_id'])]
pf_with_names = pf_sofia.merge(features[['feature_id', 'name_en']], on='feature_id')

# Pivot таблица
feat_pivot = pf_with_names.pivot_table(
    index='property_id',
    columns='name_en',
    aggfunc='size',
    fill_value=0
).clip(upper=1)  # binary 0/1

# Обединяване с основните данни
df = df.merge(feat_pivot, on='property_id', how='left')
df[feat_pivot.columns] = df[feat_pivot.columns].fillna(0).astype(int)

print(f'Финален dataset: {df.shape[0]} реда, {df.shape[1]} колони')
print(f'Добавени удобства: {len(feat_pivot.columns)}')

In [ ]:
# Добавяне на цена на квадратен метър
df['area_m2'] = pd.to_numeric(df['area_m2'], errors='coerce')
df['price_per_m2'] = df['price'] / df['area_m2']

# Премахване на ненужни колони
drop_cols = ['property_id', 'geo_id', 'property_type_id', 'construction_type_id', 'transaction_type']
df.drop(columns=drop_cols, inplace=True)

# Филтриране на нереалистични цени
before = len(df)
df = df[(df['price'] >= 20000) & (df['price'] <= 1_000_000)].copy()
# Премахване на нереалистични цени на м2 (под 200 или над 10000 EUR/m2)
df = df[(df['price_per_m2'] >= 200) & (df['price_per_m2'] <= 10000)].copy()
print(f'Премахнати outliers: {before - len(df)}')
print(f'Финален размер: {len(df)} реда')

df.head()

## 4. Изследване на данните (EDA)

In [ ]:
# Целева променлива — price (EUR)
print('Статистика на цените на апартаменти (EUR):')
print(df['price'].describe())
print(f'\nМедиана: {df["price"].median():,.0f} EUR')
print(f'Средна стойност: {df["price"].mean():,.0f} EUR')
print(f'\nСтатистика на цените на м² (EUR/m²):')
print(df['price_per_m2'].describe())

In [ ]:
# Разпределение на цените
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['price'], bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Разпределение на цените на апартаменти')
axes[0].set_xlabel('Цена (EUR)')
axes[0].set_ylabel('Брой')

axes[1].hist(np.log1p(df['price']), bins=50, color='coral', edgecolor='black')
axes[1].set_title('Разпределение на log(Цена)')
axes[1].set_xlabel('log(Цена)')
axes[1].set_ylabel('Брой')

plt.tight_layout()
plt.show()

In [ ]:
# Топ 10 корелации с цената
numeric_cols = df.select_dtypes(include=[np.number]).columns
correlations = df[numeric_cols].corr()['price'].drop('price').sort_values(ascending=False)

print('Топ 10 положителни корелации с цената:')
print(correlations.head(10).to_string())
print('\nТоп 5 отрицателни корелации:')
print(correlations.tail(5).to_string())

In [ ]:
# Средна цена по тип апартамент
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

type_prices = df.groupby('property_type')['price'].median().sort_values(ascending=True)
type_prices.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Медианна цена по тип апартамент')
axes[0].set_xlabel('Цена (EUR)')

# Средна цена по тип строителство
const_prices = df.groupby('construction_type')['price'].median().sort_values(ascending=True)
const_prices.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Медианна цена по тип строителство')
axes[1].set_xlabel('Цена (EUR)')

plt.tight_layout()
plt.show()

In [ ]:
# Цена спрямо площ
fig, ax = plt.subplots(figsize=(10, 6))
valid_area = df[(df['area_m2'] > 0) & (df['area_m2'] < 500)]
ax.scatter(valid_area['area_m2'], valid_area['price'], alpha=0.2, s=10, color='steelblue')
ax.set_xlabel('Площ (m2)', fontsize=12)
ax.set_ylabel('Цена (EUR)', fontsize=12)
ax.set_title('Цена спрямо площ на апартаментите в София', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Липсващи стойности
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f'Колони с липсващи стойности: {len(missing)}')
print()
print(missing.to_string())

## 5. Подготовка на данните с FastAI

In [ ]:
# Запълване на липсващи стойности преди log трансформация
for col in ['area_m2', 'floor', 'total_floors', 'year_built', 'bedrooms', 'price_per_m2']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median())

for col in ['property_type', 'property_category', 'construction_type',
            'construction_status', 'neighborhood', 'gas', 'tec']:
    if col in df.columns:
        df[col] = df[col].fillna('unknown')

# Log трансформация на целевата променлива
dep_var = 'price'
df['price'] = np.log1p(df['price'])

In [ ]:
# Определяне на категорийни и числови колони
cat_names = ['property_type', 'construction_type',
             'construction_status', 'neighborhood', 'gas', 'tec']
# Само колони, които съществуват в df
cat_names = [c for c in cat_names if c in df.columns]

cont_names = ['area_m2', 'floor', 'total_floors', 'year_built', 'bedrooms', 'price_per_m2']
# Добавяме удобствата като числови
feature_cols = [c for c in df.columns if c not in cat_names + cont_names + [dep_var, 'property_category']]
# Филтрираме само числови feature колони (удобствата)
feature_cols = [c for c in feature_cols if df[c].dtype in ['int64', 'float64', 'int32']]
cont_names = cont_names + feature_cols
cont_names = [c for c in cont_names if c in df.columns]

print(f'Категорийни колони ({len(cat_names)}): {cat_names}')
print(f'Числови колони ({len(cont_names)}): {cont_names[:10]}...')
print(f'Общо характеристики: {len(cat_names) + len(cont_names)}')

In [ ]:
# FastAI preprocessing
procs = [Categorify, FillMissing, Normalize]

# Разделяне на training и validation set (80/20)
splits = RandomSplitter(valid_pct=0.2, seed=42)(range_of(df))

print(f'Training: {len(splits[0])} примера')
print(f'Validation: {len(splits[1])} примера')

In [ ]:
# Създаване на TabularPandas обект
to = TabularPandas(
    df,
    procs=procs,
    cat_names=cat_names,
    cont_names=cont_names,
    y_names=dep_var,
    splits=splits,
    y_block=RegressionBlock()
)

print(f'Training items: {len(to.train)}')
print(f'Validation items: {len(to.valid)}')
to.show(3)

In [ ]:
# DataLoaders
dls = to.dataloaders(bs=64)
dls.show_batch()

## 6. Обучение на модела

In [ ]:
# Създаване на модела
# layers=[400, 200] — два скрити слоя
# ps=[0.3, 0.3] — 30% Dropout за регуларизация
# wd=0.1 — weight decay за допълнителна регуларизация
learn = tabular_learner(
    dls,
    layers=[400, 200],
    metrics=[rmse, mae],
    config=tabular_config(ps=[0.3, 0.3], embed_p=0.1),
    y_range=(df['price'].min() * 0.9, df['price'].max() * 1.1),
    wd=0.1
)

print('Архитектура на модела:')
print(learn.model)

In [ ]:
# Търсене на оптимален learning rate
learn.lr_find()

In [ ]:
# Обучение — 15 епохи с OneCycleLR scheduler
learn.fit_one_cycle(15, lr_max=1e-3)

In [ ]:
# Графика на загубата (loss) по време на обучението
learn.recorder.plot_loss()

## 7. Анализ на Training vs Validation Loss

In [ ]:
# Детайлен анализ на Training vs Validation Loss
train_losses = [x.item() for x in learn.recorder.losses]
# recorder.values съдържа [valid_loss, rmse, mae] за всяка епоха
valid_losses = [x[0] if isinstance(x[0], float) else x[0].item() for x in learn.recorder.values]
n_epochs = len(valid_losses)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Validation Loss по епохи
axes[0].plot(range(1, n_epochs + 1), valid_losses, 'o-', color='coral', label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Епоха', fontsize=12)
axes[0].set_ylabel('Loss (MSE)', fontsize=12)
axes[0].set_title('Validation Loss по епохи', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Training vs Validation Loss
batches_per_epoch = len(train_losses) // n_epochs
epoch_train_losses = []
for i in range(n_epochs):
    start = i * batches_per_epoch
    end = start + batches_per_epoch
    epoch_train_losses.append(np.mean(train_losses[start:end]))

axes[1].plot(range(1, n_epochs + 1), epoch_train_losses, 'o-', color='steelblue', label='Training Loss', linewidth=2, markersize=4)
axes[1].plot(range(1, n_epochs + 1), valid_losses, 's-', color='coral', label='Validation Loss', linewidth=2, markersize=4)
axes[1].set_xlabel('Епоха', fontsize=12)
axes[1].set_ylabel('Loss (MSE)', fontsize=12)
axes[1].set_title('Training vs Validation Loss', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Числов отчет
print(f'Начална Validation Loss (епоха 1): {valid_losses[0]:.6f}')
print(f'Крайна Validation Loss (епоха {n_epochs}): {valid_losses[-1]:.6f}')
print(f'Най-ниска Validation Loss: {min(valid_losses):.6f} (епоха {valid_losses.index(min(valid_losses)) + 1})')
print(f'Крайна Training Loss: {epoch_train_losses[-1]:.6f}')
print(f'\nРазлика train/valid: {valid_losses[-1] - epoch_train_losses[-1]:.6f}')
if valid_losses[-1] > epoch_train_losses[-1] * 1.5:
    print('Възможно overfitting — validation loss е значително по-висока от training loss')
else:
    print('Моделът генерализира добре — няма сериозно overfitting')

## 8. Оценка на модела

In [ ]:
# Предсказания на валидационното множество
preds, targets = learn.get_preds()

# Обратно от log трансформацията за реални цени
preds_real = np.expm1(preds.numpy().flatten())
targets_real = np.expm1(targets.numpy().flatten())

# Метрики
rmse_val = np.sqrt(mean_squared_error(targets_real, preds_real))
mae_val = mean_absolute_error(targets_real, preds_real)
r2_val = r2_score(targets_real, preds_real)

print('=' * 50)
print('РЕЗУЛТАТИ НА ВАЛИДАЦИОННОТО МНОЖЕСТВО')
print('=' * 50)
print(f'RMSE: {rmse_val:,.2f} EUR')
print(f'MAE:  {mae_val:,.2f} EUR')
print(f'R² Score: {r2_val:.4f}')
print(f'\nСредна реална цена: {targets_real.mean():,.2f} EUR')
print(f'Средна грешка като % от средната цена: {(mae_val / targets_real.mean()) * 100:.1f}%')
print('=' * 50)

In [ ]:
# Визуализация: Предсказани vs Реални цени
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(targets_real, preds_real, alpha=0.3, s=10, color='steelblue')
max_val = max(targets_real.max(), preds_real.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Перфектно предсказване')
axes[0].set_xlabel('Реална цена (EUR)', fontsize=12)
axes[0].set_ylabel('Предсказана цена (EUR)', fontsize=12)
axes[0].set_title('Предсказани vs Реални цени на апартаменти в София', fontsize=14)
axes[0].legend(fontsize=11)

errors = preds_real - targets_real
axes[1].hist(errors, bins=50, color='coral', edgecolor='black')
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=2)
axes[1].set_xlabel('Грешка (EUR)', fontsize=12)
axes[1].set_ylabel('Брой', fontsize=12)
axes[1].set_title('Разпределение на грешките', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# Примерни предсказания
sample_idx = np.random.choice(len(preds_real), 10, replace=False)

results_df = pd.DataFrame({
    'Реална цена (EUR)': [f'{targets_real[i]:,.0f}' for i in sample_idx],
    'Предсказана цена (EUR)': [f'{preds_real[i]:,.0f}' for i in sample_idx],
    'Грешка (EUR)': [f'{preds_real[i] - targets_real[i]:+,.0f}' for i in sample_idx],
    'Грешка (%)': [f'{((preds_real[i] - targets_real[i]) / targets_real[i]) * 100:+.1f}%' for i in sample_idx]
})

print('Примерни предсказания:')
results_df

## 9. Важност на характеристиките (Feature Importance)

In [ ]:
# Permutation importance
def get_feature_importance(learn, to, cat_names, cont_names):
    """Изчисляване на feature importance чрез permutation."""
    preds_base, targets = learn.get_preds()
    base_rmse = np.sqrt(mean_squared_error(targets.numpy(), preds_base.numpy()))
    
    importances = {}
    all_features = cat_names + cont_names
    valid_df = to.valid.xs.copy()
    
    for feat in all_features:
        if feat not in valid_df.columns:
            continue
        saved = valid_df[feat].values.copy()
        valid_df[feat] = np.random.permutation(valid_df[feat].values)
        
        dl = learn.dls.test_dl(valid_df)
        preds_perm, _ = learn.get_preds(dl=dl)
        perm_rmse = np.sqrt(mean_squared_error(targets.numpy(), preds_perm.numpy()))
        
        importances[feat] = perm_rmse - base_rmse
        valid_df[feat] = saved
    
    return pd.Series(importances).sort_values(ascending=False)

print('Изчисляване на feature importance...')
fi = get_feature_importance(learn, to, cat_names, cont_names)

top_fi = fi.head(15)

plt.figure(figsize=(10, 6))
plt.barh(range(len(top_fi)), top_fi.values[::-1], color='steelblue')
plt.yticks(range(len(top_fi)), top_fi.index[::-1])
plt.xlabel('Увеличение на RMSE при разбъркване')
plt.title('Топ 15 най-важни характеристики за цената на апартаменти в София')
plt.tight_layout()
plt.show()

## 10. Запазване на модела

In [ ]:
learn.export('sofia_apartment_prices_model.pkl')
print('Моделът е запазен като sofia_apartment_prices_model.pkl')

In [ ]:
# Демонстрация на зареждане и предсказване
learn_loaded = load_learner('sofia_apartment_prices_model.pkl')

# Предсказване за примерен апартамент
test_row = df.iloc[0].copy()
test_row['price'] = 0  # скриваме реалната цена
row, clas, probs = learn_loaded.predict(test_row)

predicted_price = np.expm1(probs.item())
print(f'Предсказана цена за примерен апартамент в София: {predicted_price:,.2f} EUR')

## 11. Заключение

### Обобщение

| Параметър | Стойност |
|-----------|----------|
| **Dataset** | Bulgaria Real Estate Listings (Kaggle) — ~22 000 апартаменти за продажба в София |
| **Характеристики** | Площ, тип апартамент, тип строителство, етаж, квартал, цена на м², удобства (46 вида) |
| **Модел** | FastAI Tabular Learner (невронна мрежа с [400, 200] неврона, Dropout 30%) |
| **Preprocessing** | Categorify, FillMissing, Normalize + log трансформация на цената |
| **Обучение** | 15 епохи, OneCycleLR, lr=0.001, weight decay=0.1 |
| **Метрики** | RMSE, MAE, R² Score |

### Ключови находки

1. **Площта (area_m2)** е най-силният предиктор за цената на апартаментите в София
2. **Цена на м² (price_per_m2)** значително подобрява предсказването
3. **Кварталът (neighborhood)** има значително влияние — апартаментите в централни квартали са значително по-скъпи
4. **Типът строителство** (тухла vs панел) и **типът апартамент** (мезонет vs студио) също оказват влияние
5. **FastAI embedding слоевете** автоматично улавят разликите между кварталите и типовете апартаменти
6. **Анализът на validation loss** показва как моделът генерализира и дали има overfitting
7. **Dropout и weight decay** са ключови за предотвратяване на overfitting при този dataset

### Използвани технологии
- Python 3.x
- FastAI 2.x / PyTorch
- pandas, numpy, matplotlib, scikit-learn
- Kaggle API